In [2]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

print(df.shape)
display(df.head())b
display(df.info())

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


None

In [3]:
print(df.columns.tolist())
print(df.isnull().sum())
print(df.dtypes)
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

for col in categorical_columns:
    df[col] = df[col].fillna("NA")

for col in numerical_columns:
    df[col] = df[col].fillna(0.0)

print(df.isnull().sum().sum())

['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object
0


In [4]:
print(df["industry"].mode()[0])

technology


In [7]:
numerical_df = df.select_dtypes(include=["number"])

corr_matrix = numerical_df.corr()

display(corr_matrix.round(3))


,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000,0.161,0.123,0.229,0.190
number_of_courses_viewed,0.161,1.000,0.722,0.757,0.395
interaction_count,0.123,0.722,1.000,0.916,0.449
lead_score,0.229,0.757,0.916,1.000,0.483
converted,0.190,0.395,0.449,0.483,1.000


In [10]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

print("Full dataset:", len(df))
print("Training:", len(df_train))
print("Validation:", len(df_val))
print("Test:", len(df_test))
y_train = df_train["converted"]
y_val = df_val["converted"]
y_test = df_test["converted"]

X_train = df_train.drop(columns=["converted"])
X_val = df_val.drop(columns=["converted"])
X_test = df_test.drop(columns=["converted"])

Full dataset: 5000
Training: 3000
Validation: 1000
Test: 1000


In [11]:
from sklearn.metrics import mutual_info_score

categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

mi_scores = {}

for col in categorical_columns:
    score = mutual_info_score(
        X_train[col],
        y_train
    )
    mi_scores[col] = round(score, 2)

mi_scores = pd.Series(mi_scores).sort_values(
    ascending=False
)

display(mi_scores)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [13]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=True)

train_dict = X_train.to_dict(orient="records")
val_dict = X_val.to_dict(orient="records")

X_train_encoded = dv.fit_transform(train_dict)
X_val_encoded = dv.transform(val_dict)

print(X_train_encoded.shape)
print(X_val_encoded.shape)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_val_encoded)

accuracy = accuracy_score(y_val, y_pred)

print("Accuracy:", round(accuracy, 2))

(3000, 28)
(1000, 28)
Accuracy: 0.65


In [14]:
original_accuracy = accuracy_score(
    y_val,
    model.predict(X_val_encoded)
)

print("Original accuracy:", original_accuracy)
features_to_remove = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

for feature in features_to_remove:
    X_train_reduced = X_train.drop(columns=[feature])
    X_val_reduced = X_val.drop(columns=[feature])

    dv_reduced = DictVectorizer(sparse=True)

    X_train_reduced_encoded = dv_reduced.fit_transform(
        X_train_reduced.to_dict(orient="records")
    )

    X_val_reduced_encoded = dv_reduced.transform(
        X_val_reduced.to_dict(orient="records")
    )

    model_reduced = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_reduced.fit(
        X_train_reduced_encoded, y_train
    )

    reduced_accuracy = accuracy_score(
        y_val,
        model_reduced.predict(X_val_reduced_encoded)
    )

    difference = original_accuracy - reduced_accuracy

    print(
        feature,
        "| Accuracy:", reduced_accuracy,
        "| Difference:", difference
    )

Original accuracy: 0.645
lead_source | Accuracy: 0.642 | Difference: 0.0030000000000000027
number_of_courses_viewed | Accuracy: 0.643 | Difference: 0.0020000000000000018
interaction_count | Accuracy: 0.601 | Difference: 0.04400000000000004


In [15]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = []

for C in C_values:
    model_reg = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    )

    model_reg.fit(X_train_encoded, y_train)

    y_pred_reg = model_reg.predict(X_val_encoded)

    acc = accuracy_score(y_val, y_pred_reg)

    results.append({
        "C": C,
        "accuracy": acc,
        "rounded_accuracy": round(acc, 3)
    })

results_df = pd.DataFrame(results)
display(results_df)

,C,accuracy,rounded_accuracy
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645
